In [2]:
# Data handling
import pandas as pd
import numpy as np

# Progress bars
from tqdm import tqdm

# Hugging Face Transformers (model, tokenizer, pipelines)
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainingArguments,
    EarlyStoppingCallback,
    PegasusTokenizer,
)

# Metrics and Evaluation
from rouge_score import rouge_scorer
from sklearn.metrics import average_precision_score

# Torch for GPU
import torch

# For displaying results in notebook
from IPython.display import display

# Plotting and Visualization (optional but useful)
import matplotlib.pyplot as plt
import seaborn as sns

# Logging
import logging

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

C:\Users\Adidya\anaconda3\Lib\site-packages\transformers\utils\hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Using device: cuda


In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering

# Example: List of article texts
articles = [
    "Article text from Newspaper 1...",
    "Article text from Newspaper 2...",
    "Another article on a different event...",
    # ...
]

# 1. Embed all articles
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(articles, show_progress_bar=True)

# 2. Cluster (tune n_clusters or use distance_threshold)
clustering = AgglomerativeClustering(n_clusters=None, distance_threshold=1.0)
clusters = clustering.fit_predict(embeddings)

# 3. Organize results
from collections import defaultdict
clustered_articles = defaultdict(list)
for idx, label in enumerate(clusters):
    clustered_articles[label].append(articles[idx])

# Print groups
for cluster, arts in clustered_articles.items():
    print(f"\nCluster {cluster}:")
    for art in arts:
        print("-", art[:80])  # Print first 80 chars

ModuleNotFoundError: No module named 'sentence_transformers'

In [7]:
from datasets import load_dataset

dataset = load_dataset("multi_news", cache_dir="internship model code", trust_remote_code=True)
print(dataset)

# Access train/val/test splits:
train_data = dataset["train"]
val_data = dataset["validation"]
test_data = dataset["test"]

# Each entry:
print("Sample document:\n", train_data[0]['document'][:1000])  # First 1000 chars
print("Sample summary:\n", train_data[0]['summary'])

train.src.cleaned:   0%|          | 0.00/548M [00:00<?, ?B/s]

train.tgt:   0%|          | 0.00/58.8M [00:00<?, ?B/s]

val.src.cleaned:   0%|          | 0.00/66.9M [00:00<?, ?B/s]

val.tgt:   0%|          | 0.00/7.30M [00:00<?, ?B/s]

test.src.cleaned:   0%|          | 0.00/69.0M [00:00<?, ?B/s]

test.tgt:   0%|          | 0.00/7.31M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/44972 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5622 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5622 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['document', 'summary'],
        num_rows: 44972
    })
    validation: Dataset({
        features: ['document', 'summary'],
        num_rows: 5622
    })
    test: Dataset({
        features: ['document', 'summary'],
        num_rows: 5622
    })
})
Sample document:
 National Archives 
 
 Yes, it’s that time again, folks. It’s the first Friday of the month, when for one ever-so-brief moment the interests of Wall Street, Washington and Main Street are all aligned on one thing: Jobs. 
 
 A fresh update on the U.S. employment situation for January hits the wires at 8:30 a.m. New York time offering one of the most important snapshots on how the economy fared during the previous month. Expectations are for 203,000 new jobs to be created, according to economists polled by Dow Jones Newswires, compared to 227,000 jobs added in February. The unemployment rate is expected to hold steady at 8.3%. 
 
 Here at MarketBeat HQ, we’ll be offering c

In [11]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "allenai/PRIMERA"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

# Example: summarize first validation example
input_text = val_data[0]['document']
inputs = tokenizer(
    input_text, 
    max_length=4096,  # PRIMERA and LED support long inputs
    truncation=True,
    return_tensors="pt"
).to(device)

summary_ids = model.generate(
    **inputs, 
    max_length=256,
    num_beams=4,
    length_penalty=2.0,
    early_stopping=True,
    no_repeat_ngram_size=3
)
print("Generated summary:\n", tokenizer.decode(summary_ids[0], skip_special_tokens=True))
print("\nReference summary:\n", val_data[0]['summary'])

Generated summary:
 Whether a sign of a good read; or a comment on the 'pulp' nature of some genres of fiction, the Oxfam second-hand book charts have remained in The Da Vinci Code author's favour for the past four years.                                   Dan Brown has topped Oxfam's 'most donated' list again, his fourth consecutive year. Having sold more than 80 million copies of The Da Jinci Code and had all four of his novels on the New York Times bestseller list in the same week, it's hardly surprising that Brown's hefty tomes are being donated to charity by readers keen to make some room on their shelves. �士                  Click here or on "View Gallery" to see both charts in pictures ||||| A woman reads a copy of the newly released book ''The Lost Symbol'' by Dan Brown, at a speed reading book launch event in Sydney, September 15, 2009. REUTERS/Tim Wimborne                  (Reporting by Alexandria Sage; editing by Carol Bishopric)    α  β  β The Oxfam shop in Swansea has been 

In [13]:
from tqdm import tqdm

N = 100  # Number of validation samples (increase for better benchmarking)
generated = []
references = []

for i in tqdm(range(N), desc="Summarizing"):
    input_text = val_data[i]['document']
    ref = val_data[i]['summary']
    inputs = tokenizer(
        input_text,
        max_length=4096,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    summary_ids = model.generate(
        **inputs,
        max_length=256,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True,
        no_repeat_ngram_size=3
    )
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    generated.append(summary)
    references.append(ref)

Summarizing: 100%|██████████| 100/100 [03:34<00:00,  2.15s/it]


In [16]:
from rouge_score import rouge_scorer
import numpy as np
import pandas as pd

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rouge1 = []
rouge2 = []
rougel = []

for ref, pred in zip(references, generated):
    scores = scorer.score(ref, pred)
    rouge1.append(scores['rouge1'].fmeasure)
    rouge2.append(scores['rouge2'].fmeasure)
    rougel.append(scores['rougeL'].fmeasure)

results_df = pd.DataFrame({
    "ROUGE-1": rouge1,
    "ROUGE-2": rouge2,
    "ROUGE-L": rougel
})

print("Average ROUGE-1:", np.mean(rouge1))
print("Average ROUGE-2:", np.mean(rouge2))
print("Average ROUGE-L:", np.mean(rougel))
display(results_df.describe())

Average ROUGE-1: 0.359612713139154
Average ROUGE-2: 0.11449982895244687
Average ROUGE-L: 0.18440836298395294


,ROUGE-1,ROUGE-2,ROUGE-L
count,100.000000,100.000000,100.000000
mean,0.359613,0.114500,0.184408
std,0.135396,0.097231,0.088684
min,0.000000,0.000000,0.000000
25%,0.287037,0.044904,0.133624
50%,0.383229,0.101654,0.169644
75%,0.449006,0.143506,0.216008
max,0.653465,0.487562,0.491566
